# BBM → Mushroom Observer cross-reference audit

**Question 1 — how many of our fungal records are cross-referenced to Mushroom Observer (MO), and how many are linked *bidirectionally* (both databases cite the same identifier)?**

**Data provenance:** `data/bbm_records.csv` is the full `collectionobject` table for the BBM fungal collection, pulled by `scripts/get_bbm_records.py` (collection-object fields only — see caveats at the end).

In [1]:
import sys
from pathlib import Path

# find the repo root (folder containing scripts/config.py) and import the audit module
ROOT = Path.cwd()
while not (ROOT / "scripts" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

import audit_mo_links as audit

print("repo root :", ROOT)
print("input CSV :", audit.INPUT)

repo root : /Users/wfrankel/Desktop/breakdowns_DES
input CSV : /Users/wfrankel/Desktop/breakdowns_DES/data/bbm_records.csv


## 1. How many of our records cite an MO number?

`scan_bbm_csv` reads every row of the BBM CSV, concatenates **all columns**, and extracts any MO reference (`MO # 82752`, `MUOB 12345`, or a mushroomobserver.org URL).

In [2]:
ref_map, n_rows, n_with_ref = audit.scan_bbm_csv(str(audit.INPUT))
n_ids = len(ref_map)

print(f"BBM fungal collection objects scanned : {n_rows}")
print(f"records citing an MO number           : {n_with_ref}  ({100*n_with_ref/n_rows:.2f}%)")
print(f"distinct MO ids cited                 : {n_ids}")

BBM fungal collection objects scanned : 34856
records citing an MO number           : 20  (0.06%)
distinct MO ids cited                 : 19


## 2. Do those references resolve on MO, and are they bidirectional?

`audit_links()` looks up each cited MO id on Mushroom Observer (batched), and classifies the link:

- **bidirectional** — we cite `MO # X` *and* MO observation X cites our catalog number back
- **unidirectional (BBM→MO)** — we cite it, MO obs exists, but doesn't cite us
- **dangling** — we cite it, but the id doesn't resolve on MO

In [4]:
res = audit.audit_links()
counts = res["counts"]
on_mo = counts["bidirectional"] + counts["unidirectional_bbm_to_mo"]

print(f"distinct MO ids cited by BBM : {res['n_ids']}")
print(f"  on MO (resolve)            : {on_mo}")
print(f"    bidirectional           : {counts['bidirectional']}")
print(f"    unidirectional (BBM->MO): {counts['unidirectional_bbm_to_mo']}")
print(f"  dangling (don't resolve)   : {counts['dangling']}")

distinct MO ids cited by BBM : 19
  on MO (resolve)            : 19
    bidirectional           : 17
    unidirectional (BBM->MO): 2
  dangling (don't resolve)   : 0


The reverse reference is found in MO's free-text `notes` field (e.g. `"Herbarium Specimen: UBC F22976"`), **not** the structured `herbarium_records` field (which is `null`). Identifiers buried in prose on both sides (category #2 on the paper)

In [5]:
import pandas as pd
pd.DataFrame(res["rows"])

,muob_id,our_catalogs,mo_exists,mo_cites_us_back,classification,mo_url,mo_consensus_name
0,66139,F023000; F23000,True,False,unidirectional_bbm_to_mo,https://mushroomobserver.org/66139,Russula densifolia
1,69671,F023058; F23058,True,True,bidirectional,https://mushroomobserver.org/69671,Russula cremoricolor
2,71507,F023007; F23007,True,True,bidirectional,https://mushroomobserver.org/71507,Russula raoultii
3,82705,F023033; F023090; F23033; F23090,True,True,bidirectional,https://mushroomobserver.org/82705,Russula abietina
4,82752,F022976; F22976,True,True,bidirectional,https://mushroomobserver.org/82752,Russula fragrantissima group
5,82916,F023062; F23062,True,True,bidirectional,https://mushroomobserver.org/82916,Russula brunneoviolacea
6,83025,F022978; F22978,True,False,unidirectional_bbm_to_mo,https://mushroomobserver.org/83025,Russula mordax
7,83026,F022977; F22977,True,True,bidirectional,https://mushroomobserver.org/83026,Russula queletii
8,83437,F023037; F23037,True,True,bidirectional,https://mushroomobserver.org/83437,Russula murrillii
9,85109,F023010; F23010,True,True,bidirectional,https://mushroomobserver.org/85109,Russula stuntzii
